# Window Aggregation in Spark Structured Streaming


Aula https://customer-academy.databricks.com/learn/courses/3959/stream-processing-and-analysis-with-apache-spark/lessons/36564/demo-window-aggregation-in-spark-structured-streaming?hash=3e063f193bec2e8ca7cdf56ffbdced1657bcc087&generated_by=409740


databricks free: https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/3454792986143832?o=7474657872577658

# 3.2 Demo - Window Aggregation in Spark Structured Streaming

**Duration:** ~25 minutes

This demo applies stateful stream processing to the Bakehouse orders and payments file streams: stateful aggregations, tumbling and sliding windows, stream-static and stream-stream joins, and watermarks for late data.

## Learning Objectives

By the end of this demo, you will be able to:
- ...
```

## B. Setup and Data Sources

We create two streaming DataFrames that we will use throughout the demo:

1. An **orders** stream containing order events
2. A **payments** stream containing payment events for those orders

Both get an event-time timestamp column derived from their unix epoch columns. Recall from Required Setup that payment timestamps are offset from order timestamps by up to 3 minutes, so the two streams are realistically out of step.
```

### Create the Orders and Payments Streaming DataFrames (início visível)


In [0]:
# Create the Orders and Payments Streaming DataFrames
from pyspark.sql.functions import *
from pyspark.sql.types import *

catalog = spark.sql("SELECT current_catalog()").first()[0]
volume_root = f"/Volumes/{catalog}/spark_stream/stream_source"

orders_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("franchise_id", LongType(), True),
    StructField("order_timestamp", LongType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", LongType(), True),
    StructField("total_price", LongType(), True),
])

payments_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("payment_method", StringType(), True),
    StructField("card_number", LongType(), True),
    StructField("amount", LongType(), True),
    StructField("payment_timestamp", LongType(), True),
])

orders_stream = (
    spark.readStream
    .format("json")
    .schema(orders_schema)
    .option("maxFilesPerTrigger", 1)
    .load(f"{volume_root}/orders")
)

payments_stream = (
    spark.readStream
    .format("json")
    .schema(payments_schema)
    .option("maxFilesPerTrigger", 1)
    .load(f"{volume_root}/payments")
)

order_events = orders_stream.withColumn(
    "order_time", to_timestamp(from_unixtime(col("order_timestamp")))
)
payment_events = payments_stream.withColumn(
    "payment_time", to_timestamp(from_unixtime(col("payment_timestamp")))
)

print(f"orders_stream is streaming: {orders_stream.isStreaming}")
print(f"payments_stream is streaming: {payments_stream.isStreaming}")

orders_stream is streaming: True
payments_stream is streaming: True


## C. Stateless vs Stateful Operations

Let's contrast the two in practice:

- **Stateless operations:** process each record independently (e.g. `select`, `filter`, `withColumn`)
- **Stateful operations:** maintain information across batches (e.g. `groupBy`, `join`)

### Stateless Operation Example

The event-time conversion above was stateless - here is the stream with one more derived column. Each row is computed from that row alone.

In [0]:
# Display a Stateless Transformation
display(
    order_events.withColumn("is_bulk_order", col("quantity") >= 24),
    checkpointLocation="/Volumes/workspace/spark_stream/stream_checkpoints/stateless_display",
)

Checkpointing to /Volumes/workspace/spark_stream/stream_checkpoints/stateless_display


In [0]:

# Stop Streaming Queries
for query in spark.streams.active:
    print(f"stopping {query.name}")
    query.stop()



In [0]:
# Display the order_events streaming DataFrame
# display() on a streaming DataFrame needs a checkpointLocation to track the stream
orders_query = (
    order_events.writeStream
    .format("memory")
    .option("checkpointLocation", "/Volumes/workspace/spark_stream/stream_checkpoints/orders_view_memory")
    .queryName("orders_view")
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

import time

time.sleep(15)  # wait for a few micro-batches to arrive

display(spark.sql("SELECT * FROM orders_view ORDER BY order_timestamp"))


order_id,customer_id,franchise_id,order_timestamp,product,quantity,total_price,order_time
101,5001,10,1795280400,Croissant,12,42,2026-11-21T17:00:00.000Z
102,5002,10,1795280460,Sourdough Loaf,30,180,2026-11-21T17:01:00.000Z
103,5003,12,1795280520,Espresso,24,72,2026-11-21T17:02:00.000Z


### Stateful Operation Example


Counting orders by product maintains a running count per group across every micro-batch - that running count is state. Watch the counts grow as files are picked up. Because the result table is continuously revised, this aggregation runs in `complete` output mode when written to a sink.

In [0]:
# Display a Stateful Aggregation
product_counts = (
    order_events
    .groupBy("product")
    .count()
    .orderBy(col("count").desc())
)

display(
    product_counts,
    checkpointLocation="/Volumes/workspace/spark_stream/stream_checkpoints/product_counts_display",
)

Checkpointing to /Volumes/workspace/spark_stream/stream_checkpoints/product_counts_display


In [0]:
for query in spark.streams.active:
    print(f"stopping {query.name}")
    query.stop()

In [0]:
# Display the product_counts result via memory sink
# The streaming aggregation is written to an in-memory table in "complete" output mode
# (aggregations without watermark only support complete mode), then read back with SQL.

for query in spark.streams.active:
    if query.name == "product_counts_view":
        print(f"stopping {query.name}")
        query.stop()

product_counts = (
    order_events
    .groupBy("product")
    .count()
)

product_counts_query = (
    product_counts.writeStream
    .format("memory")
    .queryName("product_counts_view")
    .option("checkpointLocation", "/Volumes/workspace/spark_stream/stream_checkpoints/product_counts_memory")
    .outputMode("complete")
    .trigger(availableNow=True)
    .start()
)

import time
time.sleep(15)  # wait for the micro-batches to aggregate the files

display(spark.sql("SELECT * FROM product_counts_view ORDER BY count DESC"))

product,count
Sourdough Loaf,1
Croissant,1


## D. Window Operations

Window operations let us aggregate over slices of event time rather than over the whole unbounded stream. We will demonstrate:

- Tumbling windows (fixed, non-overlapping)
- Sliding windows (overlapping)

The Bakehouse order events span several weeks, so we use day-scale windows; the mechanics are identical at any granularity.


### Tumbling Window Example

Count orders and sum revenue per 1-day tumbling window. The `window()` function inside `groupBy()` produces a `window` struct column with `start` and `end` fields.

In [0]:
# Count Orders per 1-Day Tumbling Window
# Demo speed-ups: cap shuffle partitions and subsample the input so the query drains in seconds
spark.conf.set("spark.sql.shuffle.partitions", 8)

for query in spark.streams.active:
    if query.name == "tumbling_window_counts":
        query.stop()

tumbling_windows = (
    order_events
    .filter(col("order_id") % 10 == 0)  # ~10% sample, still populates every window
    .groupBy(window(col("order_time"), "1 day"))
    .agg(
        count("*").alias("order_count"),
        sum("total_price").alias("revenue"),
    )
)

tumbling_window_query = (
    tumbling_windows.writeStream
    .format("memory")
    .option("checkpointLocation", "/Volumes/workspace/spark_stream/stream_checkpoints/tumbling_window_counts")
    .queryName("tumbling_window_counts")
    .outputMode("complete")
    .trigger(availableNow=True)
    .start()
)

tumbling_window_query.awaitTermination()


In [0]:
%sql
-- Query the Tumbling Window Results
SELECT
    window.start AS window_start,
    window.end AS window_end,
    order_count,
    revenue
FROM tumbling_window_counts
ORDER BY window_start;

window_start,window_end,order_count,revenue


In [0]:
# Rerun the Tumbling Window query with a FRESH checkpoint and show the results
# The empty result usually happens because the checkpoint already had committed offsets,
# so no new micro-batch ran and the in-memory table was never repopulated.

spark.conf.set("spark.sql.shuffle.partitions", 8)

for query in spark.streams.active:
    if query.name == "tumbling_window_counts":
        print(f"stopping {query.name}")
        query.stop()

# Remove the stale checkpoint so all input files get reprocessed
dbutils.fs.rm(
    "/Volumes/workspace/spark_stream/stream_checkpoints/tumbling_window_counts",
    recurse=True,
)

tumbling_windows = (
    order_events
   # .filter(col("order_id") % 10 == 0)  # ~10% sample # retirei o filtro
    .groupBy(window(col("order_time"), "1 day"))
    .agg(
        count("*").alias("order_count"),
        sum("total_price").alias("revenue"),
    )
)

tumbling_window_query = (
    tumbling_windows.writeStream
    .format("memory")
    .option("checkpointLocation", "/Volumes/workspace/spark_stream/stream_checkpoints/tumbling_window_counts")
    .queryName("tumbling_window_counts")
    .outputMode("complete")
    .trigger(availableNow=True)
    .start()
)

tumbling_window_query.awaitTermination()

# Now query the in-memory table
result_df = spark.sql("""
    SELECT
        window.start AS window_start,
        window.end AS window_end,
        order_count,
        revenue
    FROM tumbling_window_counts
    ORDER BY window_start
""")

print(f"rows returned: {result_df.count()}")
display(result_df)

rows returned: 1


window_start,window_end,order_count,revenue
2026-11-21T00:00:00.000Z,2026-11-22T00:00:00.000Z,3,294


In [0]:
# stop after a few cycles
for query in spark.streams.active:
    print(f"stopping {query.name}")
    query.stop()

### Sliding Window Example

Now count orders per 1-day window sliding every 6 hours. Each event lands in four overlapping windows, so the totals deliberately overlap - that is the point: every 6 hours you get a fresh trailing 24-hour reading.

In [0]:
# Count Orders per Sliding Window (1 Day, Sliding Every 6 Hours)
spark.conf.set("spark.sql.shuffle.partitions", 8)

for query in spark.streams.active:
    if query.name == "sliding_window_counts":
        query.stop()

sliding_windows = (
    order_events
    #.filter(col("order_id") % 10 == 0)  # ~10% sample for a punchy demo
    .groupBy(window(col("order_time"), "1 day", "6 hours"))
    .agg(count("*").alias("order_count"))
)

sliding_window_query = (
    sliding_windows.writeStream
    .format("memory")
    .option("checkpointLocation", "/Volumes/workspace/spark_stream/stream_checkpoints/sliding_window_counts")
    .queryName("sliding_window_counts")
    .outputMode("complete")
    .trigger(availableNow=True)
    .start()
)

sliding_window_query.awaitTermination()




In [0]:
# Now query the in-memory table
result_df = spark.sql("""
    SELECT
        window.start AS window_start,
        window.end AS window_end,
        order_count
    FROM sliding_window_counts
    ORDER BY window_start
""")

print(f"rows returned: {result_df.count()}")
display(result_df)

rows returned: 4


window_start,window_end,order_count
2026-11-20T18:00:00.000Z,2026-11-21T18:00:00.000Z,3
2026-11-21T00:00:00.000Z,2026-11-22T00:00:00.000Z,3
2026-11-21T06:00:00.000Z,2026-11-22T06:00:00.000Z,3
2026-11-21T12:00:00.000Z,2026-11-22T12:00:00.000Z,3


In [0]:
%sql
-- Query the Sliding Window Results
SELECT
    window.start AS window_start,
    window.end AS window_end,
    order_count
FROM sliding_window_counts
ORDER BY window_start
LIMIT 20;

window_start,window_end,order_count
2026-11-20T18:00:00.000Z,2026-11-21T18:00:00.000Z,3
2026-11-21T00:00:00.000Z,2026-11-22T00:00:00.000Z,3
2026-11-21T06:00:00.000Z,2026-11-22T06:00:00.000Z,3
2026-11-21T12:00:00.000Z,2026-11-22T12:00:00.000Z,3


In [0]:
# stop after a few cycles
for query in spark.streams.active:
    print(f"stopping {query.name}")
    query.stop()

### E. Streaming Joins


Let's enrich and correlate the streams.


### Stream-Static Join

A streaming DataFrame joins against a static DataFrame with no special handling - the static side is simply looked up for each micro-batch. Here we enrich the orders stream with customer details from the Bakehouse customers sample table.


In [0]:
# Enrich the Orders Stream with Static Customer Data
customers_static = (
    spark.read.table("samples.bakehouse.sales_customers")
    .select(
        col("customerID").alias("customer_id"),
        concat_ws(" ", col("first_name"), col("last_name")).alias("customer_name"),
        col("city"),
        col("country"),
    )
)


In [0]:
enriched_orders = order_events.join(customers_static, "customer_id")

display(
    enriched_orders.select("order_id", "order_time", "product", "customer_name", "city", "country"),
    checkpointLocation="/Volumes/workspace/spark_stream/stream_checkpoints/enriched_orders_display",
)

Checkpointing to /Volumes/workspace/spark_stream/stream_checkpoints/enriched_orders_display


In [0]:
customers_static.show()

+-----------+------------------+----------------+---------+
|customer_id|     customer_name|            city|  country|
+-----------+------------------+----------------+---------+
|    2000259|     Kayla Barrett|  Kathrynborough|    Japan|
|    2000260|       Amanda Reed|  East Catherine|    Japan|
|    2000261|     Steven Tanner|    Williamshire|    Japan|
|    2000262|   Jennifer Forbes|   Delacruzville|Australia|
|    2000263|    Kenneth Berger|      West Wendy|Australia|
|    2000264|     James Edwards|       Sherryton|    Japan|
|    2000265|    Ann Montgomery|       East Todd|Australia|
|    2000266|     Heather Moore|     Josephburgh|Australia|
|    2000267|     Joseph Graham|      Kelseyview|Australia|
|    2000268|      Lisa Ramirez| West Deniseland|    Japan|
|    2000269|      Sierra Jones|    Michaelmouth|      USA|
|    2000270|     Tyler Jackson|   Port Brianton|Australia|
|    2000271|      Rachael Lowe|   Willieborough|Australia|
|    2000272|     Shelly Rogers|     New

In [0]:
# Ver o conteúdo de enriched_orders
# O display() de streaming (célula anterior) não salva a saída em tabela nenhuma,
# então reconstruímos o mesmo join e materializamos num sink de memória para exibir.

for query in spark.streams.active:
    if query.name == "enriched_orders_view":
        print(f"stopping {query.name}")
        query.stop()

# checkpoint novo para garantir que os arquivos sejam reprocessados
dbutils.fs.rm(
    "/Volumes/workspace/spark_stream/stream_checkpoints/enriched_orders_view",
    recurse=True,
)

enriched_orders = order_events.join(customers_static, "customer_id")

enriched_orders_query = (
    enriched_orders.writeStream
    .format("memory")
    .queryName("enriched_orders_view")
    .option("checkpointLocation", "/Volumes/workspace/spark_stream/stream_checkpoints/enriched_orders_view")
    .outputMode("append")
    .trigger(availableNow=True)
    .start()
)

enriched_orders_query.awaitTermination()

result_df = spark.sql("""
    SELECT order_id, order_time, product, total_price, customer_name, city, country
    FROM enriched_orders_view
    ORDER BY order_time
""")

print(f"rows returned: {result_df.count()}")
display(result_df)


rows returned: 0


order_id,order_time,product,total_price,customer_name,city,country


In [0]:
# stop after a few cycles
for query in spark.streams.active:
    print(f"stopping {query.name}")
    query.stop()

### Stream-Stream Join

Joining two streams requires Spark to buffer rows from both sides as state, since a match may arrive later on either side. To keep that state bounded we:

1. Add a watermark to both streams
2. Bound the match interval in the join condition - a payment must occur within 10 minutes of its order

Without these, the join state grows forever.

In [0]:
### Join the Orders and Payments Streams with Watermarks
# Join the Orders and Payments Streams with Watermarks
for query in spark.streams.active:
    if query.name == "order_payment_join":
        query.stop()

orders_wm = order_events.withWatermark("order_time", "10 minutes")
payments_wm = payment_events.withWatermark("payment_time", "10 minutes")

order_payment_join = orders_wm.join(
    payments_wm,
    (orders_wm.order_id == payments_wm.order_id)
    & (payments_wm.payment_time >= orders_wm.order_time)
    & (payments_wm.payment_time <= orders_wm.order_time + expr("INTERVAL 10 MINUTES")),
)

join_query = (
    order_payment_join
    .select(
        orders_wm.order_id, "product", "quantity",
        "payment_method", "amount", "order_time", "payment_time",
    )
    .writeStream
    .format("memory")
    .option("checkpointLocation", "/Volumes/workspace/spark_stream/stream_checkpoints/order_payment_join")
    .outputMode("append")
    .queryName("order_payment_join")
    .trigger(availableNow=True)
    .start()
)

join_query.awaitTermination()

In [0]:
%sql
-- Query the Joined Streams
SELECT order_id, product, payment_method, amount, order_time, payment_time
FROM order_payment_join
ORDER BY order_time
LIMIT 20;


order_id,product,payment_method,amount,order_time,payment_time
101,Croissant,credit_card,42,2026-11-21T17:00:00.000Z,2026-11-21T17:02:43.000Z
102,Sourdough Loaf,debit_card,180,2026-11-21T17:01:00.000Z,2026-11-21T17:02:10.000Z
103,Espresso,credit_card,72,2026-11-21T17:02:00.000Z,2026-11-21T17:04:53.000Z


In [0]:
### Stop Streaming Queries


# stop after a few cycles
for query in spark.streams.active:
    print(f"stopping {query.name}")
    query.stop()


## F. Handling Late Data with Watermarks

Watermarks also matter for windowed aggregations:

- The watermark defines how long to wait for late events before a window is finalized
- We use `update` output mode so only the windows that changed in each micro-batch are emitted
- Once the watermark passes a window's end, its state is dropped from the state store - preventing unbounded state growth

In [0]:
# Windowed Aggregation with a Watermark
for query in spark.streams.active:
    if query.name == "windowed_with_watermark":
        query.stop()


In [0]:
# Windowed Aggregation with a Watermark
# Interrompe qualquer consulta anterior com o mesmo nome para evitar conflitos ao reexecutar
for query in spark.streams.active:
    if query.name == "windowed_with_watermark":
        query.stop()

# Define um watermark de 2 horas em order_time e agrupa por janelas de 6 horas e por produto
watermarked_windows = (
    order_events
    .withWatermark("order_time", "2 hours")
    .groupBy(window(col("order_time"), "6 hours"), col("product"))
    .count()
)

# Escreve em memória no modo update com availableNow=True para processar os dados disponíveis de forma determinística
watermark_query = (
    watermarked_windows.writeStream
    .format("memory")
    .option("checkpointLocation", "/Volumes/workspace/spark_stream/stream_checkpoints/windowed_with_watermark")
    .outputMode("update")
    .queryName("windowed_with_watermark")
    .trigger(availableNow=True)
    .start()
)



In [0]:
%sql
-- Query the Watermarked Window Results
SELECT
    window.start AS window_start,
    window.end AS window_end,
    product,
    count
FROM windowed_with_watermark
ORDER BY window_start, product
LIMIT 20;

window_start,window_end,product,count
2026-11-21T12:00:00.000Z,2026-11-21T18:00:00.000Z,Croissant,1
2026-11-21T12:00:00.000Z,2026-11-21T18:00:00.000Z,Espresso,1
2026-11-21T12:00:00.000Z,2026-11-21T18:00:00.000Z,Sourdough Loaf,1



### Stop All Active Streaming Queries

```markdown
Run the cell below to stop the active streaming queries.

In [0]:
# Stop All Active Streaming Queries
# Interrompe todas as consultas de streaming ainda em execução na sessão atual
for query in spark.streams.active:
    query.stop()

# Confirma que não restam consultas ativas
print(f"Active streaming queries: {len(spark.streams.active)}")

Active streaming queries: 0


## Summary

In this demo you:

- Contrasted a stateless transformation with a stateful `groupBy` aggregation whose running counts persist across micro-batches
- Aggregated order events into 1-day tumbling windows and overlapping 1-day/6-hour sliding windows using `window()` over event time
- Enriched the orders stream with static customer data via a stream-static join
- Joined the orders and payments streams with watermarks on both sides and a bounded match interval to keep join state in check

- Combined a watermark with `update` output mode so late events are tolerated for 2 hours while old window state is dropped